In [1]:
from pathlib import Path
from PIL import Image
from fastai.vision.all import *

DATASET_PATH = Path('/kaggle/input/datasets/ranveer112/plantvillage/dataset')

Core problem:

Plants have diseases and their maintainers are not botanist. Therefore, they need a solution for diagnosis of disease.

Modality used:
Images of leaves.



In [23]:
#We model each of test, train, validation as follows



LENGTH=256
WIDTH=256
CHANNELS=3

# Define the path to your train directory
train_dir = DATASET_PATH / 'train'
test_dir = DATASET_PATH / 'test'
validation_dir = DATASET_PATH / 'validation'

# 1. Start with an empty Python list
test_x_list = []
test_y_list= []
validation_x_list = []
validation_y_list = []
training_x_list=[]
training_y_list=[]
classId=dict()

for pathToImage in train_dir.glob('**/*.JPG'):
    
    # Open the image, convert it to a tensor, and append
    img_tensor = tensor(Image.open(pathToImage))
    className = pathToImage.parent.name
    classId[className] = classId[className] if className in classId else len(classId)
    training_x_list.append(img_tensor)
    training_y_list.append(classId[className])
    # Your assertion check
    assert training_x_list[-1].shape == (LENGTH, WIDTH, CHANNELS)


for pathToImage in test_dir.glob('**/*.JPG'):
    
    # Open the image, convert it to a tensor, and append
    img_tensor = tensor(Image.open(pathToImage))
    className = pathToImage.parent.name
    test_x_list.append(img_tensor)
    test_y_list.append(classId[className])
    # Your assertion check
    assert test_x_list[-1].shape == (LENGTH, WIDTH, CHANNELS)


for pathToImage in validation_dir.glob('**/*.JPG'):
    
    # Open the image, convert it to a tensor, and append
    img_tensor = tensor(Image.open(pathToImage))
    className = pathToImage.parent.name
    validation_x_list.append(img_tensor)
    validation_y_list.append(classId[className])
    # Your assertion check
    assert validation_x_list[-1].shape == (LENGTH, WIDTH, CHANNELS)


testX=torch.stack(test_x_list, dim=0)
trainX=torch.stack(training_x_list, dim=0)
validationX=torch.stack(validation_x_list, dim=0)

testY=torch.tensor(test_y_list)
trainY=torch.tensor(training_y_list)
validationY=torch.tensor(validation_y_list)


We will do multiple models to see how they fare with each other:

a) CNN
b) Perceptron
c) Neural net without CNN
d) Random forest



In [11]:
#Let's try random forests 
from sklearn.ensemble import RandomForestClassifier


trainX_np = trainX.view(trainX.shape[0], -1).numpy()
validationX_np = validationX.view(validationX.shape[0], -1).numpy()
testX_np = testX.view(testX.shape[0], -1).numpy()

# Convert labels to numpy arrays
trainY_np = trainY.numpy()
validationY_np = validationY.numpy()
testY_np = testY.numpy()

def misclassified(m, xs, y):
    preds=m.predict(xs)
    miscount=0
    for idx in range(len(preds)):
        if preds[idx]!=y[idx]:
            miscount+=1
    return miscount/len(preds)

m = RandomForestClassifier(
    n_jobs=-1, 
    n_estimators=40,
    oob_score=True,
    random_state=40
)
m.fit(trainX_np, trainY_np)

print(f"Training error: {misclassified(m, trainX_np, trainY_np)}")
print(f"Validation error: {misclassified(m, validationX_np, validationY_np)}")
print(f"Testing error: {misclassified(m, testX_np, testY_np)}")


Training error: 0.0
Validation error: 0.4015594541910331
Testing error: 0.40497076023391815


In [31]:
#Neural net without CNN
from torch.utils.data import TensorDataset, DataLoader
from fastai.vision.all import DataLoaders
import torch.optim as optim

#Model architecture : Linear layers + RELU for non-linearity, 1 hidden layer
#30 params, 15 node output layer, 256*256*3 input layer
def batch_accuracy(xb, yb):
    probs = xb.float().softmax(dim=-1)
    preds = probs.argmax(dim=-1)
    correct = (preds == yb)
    return correct.float().mean()


simple_net = nn.Sequential(
    nn.Linear(LENGTH*WIDTH*CHANNELS,2000),
    nn.ReLU(),
    nn.Linear(2000, 15)
    
)

# 1. Flatten, cast, and normalize (keeping them as efficient PyTorch tensors)
train_X_flat = trainX.view(trainX.shape[0], -1)#valid_X_processed = validationX.view(validationX.shape[0], -1).float() / 255.0
valid_X_flat = validationX.view(validationX.shape[0], -1)

# 2. Wrap the raw integer tensors into datasets
train_dset = TensorDataset(train_X_flat, trainY)
valid_dset = TensorDataset(valid_X_flat, validationY)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
simple_net = simple_net.to(device)
# 3. Create normal PyTorch DataLoaders
train_loader = DataLoader(train_dset, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_dset, batch_size=256, shuffle=False)
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_net.parameters(), lr=0.001)
#Learning algorithm :  Adam
#Loss function cross entropy

EPOCHS=40
LEARNING_RATE=0.1
# Function to calculate validation metrics cleanly
def evaluate_loss_acc(model, loader, loss_func, device):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    total_samples = 0
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float() / 255.0
            yb = yb.to(device)
            
            preds = model(xb)
            loss = loss_func(preds, yb)
            
            total_loss += loss.item() * xb.size(0)
            total_acc += batch_accuracy(preds, yb).item() * xb.size(0)
            total_samples += xb.size(0)
                
    return total_loss / total_samples, total_acc / total_samples

# --- RUN STATISTICS BEFORE TRAINING (EPOCH 00) ---
init_train_loss, init_train_acc = evaluate_loss_acc(simple_net, train_loader, loss_func, device)
init_valid_loss, init_valid_acc = evaluate_loss_acc(simple_net, valid_loader, loss_func, device)

print(f"Epoch 00/{EPOCHS} -> "
      f"Train Loss: {init_train_loss:.4f} | Train Acc: {init_train_acc:.4f} | "
      f"Valid Loss: {init_valid_loss:.4f} | Valid Acc: {init_valid_acc:.4f}")

# --- TRAINING LOOP ---
for epoch in range(1, EPOCHS + 1):
    simple_net.train()
    
    for xb, yb in train_loader:
        xb = xb.to(device).float() / 255.0
        yb = yb.to(device)
        
        # Optimization Step
        preds = simple_net(xb)
        loss = loss_func(preds, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # Evaluate BOTH training and validation metrics exactly at the end of epoch i
    epoch_train_loss, epoch_train_acc = evaluate_loss_acc(simple_net, train_loader, loss_func, device)
    epoch_valid_loss, epoch_valid_acc = evaluate_loss_acc(simple_net, valid_loader, loss_func, device)
    
    print(f"Epoch {epoch:02d}/{EPOCHS} -> "
      f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
      f"Valid Loss: {epoch_valid_loss:.4f} | Valid Acc: {epoch_valid_acc:.4f}")

Epoch 00/40 -> Train Loss: 2.7239 | Train Acc: 0.0557 | Valid Loss: 2.7234 | Valid Acc: 0.0565
Epoch 01/40 -> Train Loss: 2.3708 | Train Acc: 0.2683 | Valid Loss: 2.3701 | Valid Acc: 0.2651
Epoch 02/40 -> Train Loss: 2.1079 | Train Acc: 0.3480 | Valid Loss: 2.1093 | Valid Acc: 0.3411
Epoch 03/40 -> Train Loss: 1.9403 | Train Acc: 0.4173 | Valid Loss: 1.9399 | Valid Acc: 0.4240
Epoch 04/40 -> Train Loss: 1.7633 | Train Acc: 0.4571 | Valid Loss: 1.7604 | Valid Acc: 0.4600
Epoch 05/40 -> Train Loss: 1.7002 | Train Acc: 0.4644 | Valid Loss: 1.6986 | Valid Acc: 0.4605
Epoch 06/40 -> Train Loss: 1.5212 | Train Acc: 0.5376 | Valid Loss: 1.5247 | Valid Acc: 0.5468
Epoch 07/40 -> Train Loss: 1.4640 | Train Acc: 0.5522 | Valid Loss: 1.4740 | Valid Acc: 0.5502
Epoch 08/40 -> Train Loss: 1.4027 | Train Acc: 0.5783 | Valid Loss: 1.4223 | Valid Acc: 0.5590
Epoch 09/40 -> Train Loss: 1.3460 | Train Acc: 0.5963 | Valid Loss: 1.3707 | Valid Acc: 0.5848
Epoch 10/40 -> Train Loss: 1.3049 | Train Acc: 0.6

In [25]:
import torch

def run_random_classifier(train_labels, valid_labels, classes_num=15, strategy="stratified"):
    """
    Simulates a random classifier over the dataset.
    Strategies:
      - "uniform": Every class has an equal 1/15 chance.
      - "stratified": Guesses based on how common the class is in training data.
    """
    total_valid = len(valid_labels)
    
    if strategy == "uniform":
        # Generate random numbers between 0 and 14 uniformly
        random_preds = torch.randint(0, classes_num, (total_valid,))
        
    elif strategy == "stratified":
        # Calculate class distribution from training data
        counts = torch.bincount(train_labels, minlength=classes_num).float()
        probabilities = counts / counts.sum()
        
        # Sample predictions following the real training distribution
        random_preds = torch.multinomial(probabilities, total_valid, replacement=True)
        
    # Calculate baseline accuracy
    correct = (random_preds == valid_labels).sum().item()
    accuracy = correct / total_valid
    
    print(f"--- Random Classifier ({strategy.upper()} strategy) ---")
    print(f"Correctly guessed: {correct}/{total_valid}")
    print(f"Baseline Validation Accuracy: {accuracy * 100:.2f}%\n")
    return accuracy


print(run_random_classifier(trainY, validationY, classes_num=15, strategy="uniform"))

print(run_random_classifier(trainY, validationY, classes_num=15, strategy="stratified"))

--- Random Classifier (UNIFORM strategy) ---
Correctly guessed: 142/2052
Baseline Validation Accuracy: 6.92%

0.06920077972709551
--- Random Classifier (STRATIFIED strategy) ---
Correctly guessed: 167/2052
Baseline Validation Accuracy: 8.14%

0.0813840155945419


In [29]:
#Now let's try CNN
#let's try with just two layers

# 2. Wrap the raw integer tensors into datasets
trainX_cnn=trainX.permute(0, 3, 1, 2)
validationX_cnn=validationX.permute(0, 3, 1, 2)
train_dset = TensorDataset(trainX_cnn, trainY)
valid_dset = TensorDataset(validationX_cnn, validationY)
# 3. Create normal PyTorch DataLoaders
train_loader = DataLoader(train_dset, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_dset, batch_size=256, shuffle=False)
cnn = sequential(
    nn.Conv2d(3, 6, stride=2, kernel_size=3, padding=1), #128*128
    nn.ReLU(),
    nn.Conv2d(6, 12, stride=2, kernel_size=3, padding=1), #64*64
    nn.ReLU(),
    nn.Conv2d(12, 24, stride=2, kernel_size=3, padding=1), #32*32
    
    nn.ReLU(),
    nn.Conv2d(24, 48, stride=2, kernel_size=3, padding=1), #16*16
    
    nn.ReLU(),
    nn.Conv2d(48, 96, stride=2, kernel_size=3, padding=1), #8*8
    
    nn.ReLU(),
    nn.Conv2d(96, 192, stride=2, kernel_size=3, padding=1), #4*4

    
    nn.ReLU(),
    nn.Conv2d(192, 384, stride=2, kernel_size=3, padding=1), #2*2
    nn.ReLU(),
    nn.Conv2d(384, 15, stride=2, kernel_size=3, padding=1), #1*1
    # Add this at the very end to turn (256, 15, 1, 1) into (256, 15)
    nn.Flatten()
)



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn = cnn.to(device)
optimizer = optim.Adam(cnn.parameters(), lr=0.001)
# --- RUN STATISTICS BEFORE TRAINING (EPOCH 00) ---
init_train_loss, init_train_acc = evaluate_loss_acc(cnn, train_loader, loss_func, device)
init_valid_loss, init_valid_acc = evaluate_loss_acc(cnn, valid_loader, loss_func, device)

print(f"Epoch 00/{EPOCHS} -> "
      f"Train Loss: {init_train_loss:.4f} | Train Acc: {init_train_acc:.4f} | "
      f"Valid Loss: {init_valid_loss:.4f} | Valid Acc: {init_valid_acc:.4f}")


for epoch in range(1, EPOCHS + 1):
    cnn.train()
    
    for xb, yb in train_loader:
        xb = xb.to(device).float() / 255.0
        yb = yb.to(device)
        
        # Optimization Step
        preds = cnn(xb)
        loss = loss_func(preds, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # Evaluate BOTH training and validation metrics exactly at the end of epoch i
    epoch_train_loss, epoch_train_acc = evaluate_loss_acc(cnn, train_loader, loss_func, device)
    epoch_valid_loss, epoch_valid_acc = evaluate_loss_acc(cnn, valid_loader, loss_func, device)
    
    print(f"Epoch {epoch:02d}/{EPOCHS} -> "
      f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
      f"Valid Loss: {epoch_valid_loss:.4f} | Valid Acc: {epoch_valid_acc:.4f}")


Epoch 00/40 -> Train Loss: 2.7085 | Train Acc: 0.0488 | Valid Loss: 2.7085 | Valid Acc: 0.0487
Epoch 01/40 -> Train Loss: 2.1991 | Train Acc: 0.2589 | Valid Loss: 2.1957 | Valid Acc: 0.2627
Epoch 02/40 -> Train Loss: 1.6601 | Train Acc: 0.4573 | Valid Loss: 1.6556 | Valid Acc: 0.4522
Epoch 03/40 -> Train Loss: 1.3383 | Train Acc: 0.5461 | Valid Loss: 1.3368 | Valid Acc: 0.5419
Epoch 04/40 -> Train Loss: 1.0960 | Train Acc: 0.6261 | Valid Loss: 1.0984 | Valid Acc: 0.6213
Epoch 05/40 -> Train Loss: 0.9018 | Train Acc: 0.6937 | Valid Loss: 0.9262 | Valid Acc: 0.6706
Epoch 06/40 -> Train Loss: 0.9141 | Train Acc: 0.6791 | Valid Loss: 0.9532 | Valid Acc: 0.6589
Epoch 07/40 -> Train Loss: 0.6974 | Train Acc: 0.7654 | Valid Loss: 0.7533 | Valid Acc: 0.7456
Epoch 08/40 -> Train Loss: 0.6979 | Train Acc: 0.7620 | Valid Loss: 0.7648 | Valid Acc: 0.7281
Epoch 09/40 -> Train Loss: 0.5594 | Train Acc: 0.8098 | Valid Loss: 0.6601 | Valid Acc: 0.7705
Epoch 10/40 -> Train Loss: 0.5325 | Train Acc: 0.8

In [30]:
trainable_params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f"Total Trainable Parameters(CNN): {trainable_params:,}")

trainable_params = sum(p.numel() for p in simple_net.parameters() if p.requires_grad)
print(f"Total Trainable Parameters(Simple Neural net): {trainable_params:,}")

Total Trainable Parameters(CNN): 937,299
Total Trainable Parameters(Simple Neural net): 393,248,015
